# perfect-map-matching — verification

# Imports

In [ ]:
import os

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

from uavloc.builds import build_dir
from uavloc.data.annotated import label_records, load_label_map
from uavloc.data.drone_frames import DERIVED, DroneFrame, load_frames
from uavloc.data.satellite import GeoImage
from uavloc.features.segmentation import CLASSES, SegmentationFeatures, label_probabilities, load_unet
from uavloc.geo import LatLon, haversine_m, offset_latlon
from uavloc.matching.benchmark import injected_priors
from uavloc.matching.correlator import CorrelatorConfig, PositionCorrelator
from uavloc.matching.identity import SatelliteAsDrone, satellite_patch_as_drone
from uavloc.plotting.correlation import CORRECTED_COLOURS, SEG_PALETTE, plot_heatmap
from uavloc.track import REPO_ROOT, experiment_dir

# Params

In [ ]:
SLUG = "perfect-map-matching"
DEVICE = "mps"
SEED = 42

EXP_001 = "001-seg-map-correction"
REPLAY_RUN = "001.3"
REPLAY_DRAWS = 12
SEG_CHECKPOINT_SAT = "pocs/village-seg-baseline/model-round3-bn-satellite.pt"
SEG_CHECKPOINT_DRONE = "pocs/village-seg-baseline/model-round3-bn-drone.pt"
DRONE_SEG_GSD_M = 0.10
MATCH_GSD_M = 0.385
SAT_IMAGE = "../archive/uav_localization_2025/data/satelites/satellite_gonchary.jpg"
SAT_TOP_LEFT = (49.762965, 24.166472)
SAT_BOTTOM_RIGHT = (49.759291, 24.174046)

SEARCH_SCALE = 1.7
FOV_W_DEG = 60.0
SAT_FEATURE_CROP_PCT = 0.03
CLASS_WEIGHTS = (0.0, 1.0, 1.0, 0.5, 0.0, 0.5)   # background, building, track, woody, field, water (proposal 002)
NOISE_M = 15.0
N_DIRECTIONS = 8
IDENTITY_FRAME_STEP = 9

LABEL_LOCATION = "birky-49p9105-23p9192"         # a known export, until the Gonchary labels arrive
LABEL_DATE = "current"
SYNTHETIC_POSES = 6
SYNTHETIC_ALTITUDE_M = 50.0
SYNTHETIC_SPREAD_M = 250.0

GONCHARY_LABELS = ("gonchary-2025", "ortho", "ua-ortho-annotated")
REQUIRED_TOP_LEFT = (49.761905, 24.167825)       # 002 proposal: every frame + farthest 15 m search window
REQUIRED_BOTTOM_RIGHT = (49.759486, 24.172788)
OVERLAY_ALPHA = 0.65

# Data

In [ ]:
FIGURES = build_dir(SLUG, create=True) / "figures"
frames = load_frames("drone-2025-easy")
satellite = GeoImage.load(REPO_ROOT / SAT_IMAGE, LatLon(*SAT_TOP_LEFT), LatLon(*SAT_BOTTOM_RIGHT))
seg_features = SegmentationFeatures(load_unet(REPO_ROOT / SEG_CHECKPOINT_DRONE, DEVICE), load_unet(REPO_ROOT / SEG_CHECKPOINT_SAT, DEVICE),
                                    satellite, "all", DRONE_SEG_GSD_M, MATCH_GSD_M, DEVICE)
cosine = CorrelatorConfig(SEARCH_SCALE, FOV_W_DEG, sat_feature_crop_pct=SAT_FEATURE_CROP_PCT)
weighted_ncc = CorrelatorConfig(SEARCH_SCALE, FOV_W_DEG, sat_feature_crop_pct=SAT_FEATURE_CROP_PCT,
                                matching="weighted-ncc", class_weights=CLASS_WEIGHTS)

# Check: cosine matching unchanged

In [ ]:
records = pd.read_csv(experiment_dir(EXP_001) / "artifacts" / f"records-{REPLAY_RUN}.csv")
replay = records[records["noise_m"] == NOISE_M].sample(REPLAY_DRAWS, random_state=SEED)
by_name = {f.name: f for f in frames}
correlator = PositionCorrelator(seg_features, cosine)
replayed = [haversine_m(correlator.correct(by_name[r.frame], LatLon(r.prior_lat, r.prior_lon)).position, by_name[r.frame].position)
            for r in replay.itertuples()]
float(np.max(np.abs(np.array(replayed) - replay["error_m"].to_numpy())))

# Check: identity

In [ ]:
def identity_errors(features, config: CorrelatorConfig, poses: list[DroneFrame]) -> pd.DataFrame:
    rows = []
    for frame in poses:
        correlator = PositionCorrelator(SatelliteAsDrone(features, frame.position, frame.yaw_deg, satellite_patch_as_drone), config)
        for bearing_deg, prior in injected_priors(frame.position, NOISE_M, N_DIRECTIONS, phase_deg=0.0):
            correction = correlator.correct(frame, prior)
            rows.append({"reachable": correlator.reachable(frame, prior), "peak_z": correction.peak_z,
                         "error_m": haversine_m(correction.position, frame.position)})
    return pd.DataFrame(rows)

In [ ]:
def summarise_identity(results: dict[str, pd.DataFrame]) -> pd.DataFrame:
    return pd.DataFrame({name: d[d["reachable"]]["error_m"].agg(["median", "max", "count"]) for name, d in results.items()}).T

In [ ]:
gonchary_poses = frames[::IDENTITY_FRAME_STEP]
summarise_identity({"round-3 map, cosine": identity_errors(seg_features, cosine, gonchary_poses),
                    "round-3 map, weighted-ncc": identity_errors(seg_features, weighted_ncc, gonchary_poses)})

# Check: label map loader

In [ ]:
labels = load_label_map(LABEL_LOCATION, LABEL_DATE)
record = next(r for r in label_records() if r["location"] == LABEL_LOCATION and r["date"] == LABEL_DATE)
image = np.asarray(Image.open(DERIVED / record["image_file"]).convert("RGB"))
labels.pixels.shape, image.shape[:2], labels.mpp, np.unique(labels.pixels)

In [ ]:
fig_labels, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].imshow(image); axes[0].set_title(f"{LABEL_LOCATION} {LABEL_DATE}")
axes[1].imshow(image); axes[1].imshow(SEG_PALETTE[np.where(labels.pixels == 255, 0, labels.pixels)], alpha=.45)
axes[1].set_title("label map as loaded, over its own image")
for ax in axes: ax.set_xticks([]); ax.set_yticks([])
fig_labels.tight_layout(); fig_labels.savefig(FIGURES / "label-map-overlay.png", dpi=80)

# Check: identity on a label map

In [ ]:
def synthetic_poses(centre: LatLon, rng: np.random.Generator) -> list[DroneFrame]:
    """Poses over a labelled village; only position, altitude and yaw matter to the identity check."""
    return [DroneFrame(f"synthetic-{i}", frames[0].image_path,
                       offset_latlon(centre, *rng.uniform(-SYNTHETIC_SPREAD_M, SYNTHETIC_SPREAD_M, 2)),
                       SYNTHETIC_ALTITUDE_M, float(rng.uniform(0, 360)), frames[0].mpp, ())
            for i in range(SYNTHETIC_POSES)]

In [ ]:
label_features = SegmentationFeatures.from_satellite_probabilities(seg_features.drone_model, label_probabilities(labels, MATCH_GSD_M),
                                                                   "all", DRONE_SEG_GSD_M, MATCH_GSD_M, DEVICE)
centre = labels.to_latlon(labels.pixels.shape[1] / 2, labels.pixels.shape[0] / 2)
poses = synthetic_poses(centre, np.random.default_rng(SEED))
summarise_identity({"label map, cosine": identity_errors(label_features, cosine, poses),
                    "label map, weighted-ncc": identity_errors(label_features, weighted_ncc, poses)})

# Check: Gonchary labels

In [ ]:
gonchary = load_label_map(*GONCHARY_LABELS)
(x0, y0), (x1, y1) = gonchary.to_px(LatLon(*REQUIRED_TOP_LEFT)), gonchary.to_px(LatLon(*REQUIRED_BOTTOM_RIGHT))
in_box = gonchary.pixels[int(y0):int(y1), int(x0):int(x1)]
gonchary.top_left == satellite.top_left and gonchary.bottom_right == satellite.bottom_right, {
    CLASSES[c]: round(float((in_box == c).mean()), 4) for c in np.unique(in_box)}

In [ ]:
ortho = np.asarray(Image.fromarray(satellite.pixels).resize(gonchary.pixels.shape[::-1], Image.BILINEAR), float)
blend = np.where(gonchary.pixels[..., None] > 0, (1 - OVERLAY_ALPHA) * ortho + OVERLAY_ALPHA * SEG_PALETTE[gonchary.pixels], ortho)
fig_gonchary, ax = plt.subplots(figsize=(14, 10))
ax.imshow(blend.astype(np.uint8))
ax.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, ec="w", lw=1.5, ls="--"))
ax.plot(*zip(*(gonchary.to_px(f.position) for f in frames)), "c.", ms=5)
ax.set_title("Gonchary hand labels over the 2025 orthophoto; box = required coverage, cyan = frames")
fig_gonchary.tight_layout(); fig_gonchary.savefig(FIGURES / "gonchary-labels.png", dpi=70)

# Figures

In [ ]:
failing = records[(records["noise_m"] == NOISE_M) & records["reachable"]].sort_values("error_m").iloc[-1]
frame, prior = by_name[failing["frame"]], LatLon(failing["prior_lat"], failing["prior_lon"])
patch_rgb = satellite.crop_rotated(prior, frame.yaw_deg, *PositionCorrelator(seg_features, cosine).patch_size_m(frame))
fig_compare, axes = plt.subplots(1, 2, figsize=(15, 4.5))
for ax, (name, config), colour in zip(axes, {"cosine": cosine, "weighted-ncc": weighted_ncc}.items(), CORRECTED_COLOURS):
    correlator = PositionCorrelator(seg_features, config)
    correction = correlator.correct(frame, prior)
    plot_heatmap(ax, patch_rgb, correction, correlator.patch_offset_m(prior, frame.position, frame.yaw_deg), colour,
                 f"{name}: {haversine_m(correction.position, frame.position):.1f} m, z {correction.peak_z:.2f}")
fig_compare.suptitle(f"round-3 map, worst reachable {REPLAY_RUN} draw ({frame.name})")
fig_compare.tight_layout(); fig_compare.savefig(FIGURES / "cosine-vs-weighted-ncc.png", dpi=90)